In [1]:
!pip install --upgrade langchain
!pip install langchain_text_splitters
!pip install langchain_community
!pip install langchain_chroma

In [3]:
import os
!pip install langchain-groq

# Install all necessary packages for LangChain components and their dependencies
from langchain_community.document_loaders import UnstructuredFileLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_classic.chains import RetrievalQA

In [ ]:
os.environ["GROQ_API_KEY"] = ""

In [5]:
#fetch the pdf url
import requests
url = "https://pythoncourses.azurewebsites.net/eBooks/Python%20Object-Oriented%20Programming.pdf"
response = requests.get(url)


In [6]:
#save the pdf to local file
with open("Python Object-Oriented Programming.pdf", "wb") as f:
    f.write(response.content)

In [7]:
!pip install "unstructured[pdf]"

In [8]:
#loading the document
loader = UnstructuredFileLoader("Python Object-Oriented Programming.pdf")

/tmp/ipykernel_9429/2124556737.py:2: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader("Python Object-Oriented Programming.pdf")


In [9]:
documents = loader.load()
documents

[Document(metadata={'source': 'Python Object-Oriented Programming.pdf'}, page_content='Python Object-Oriented Programming\n\nFourth Edition\n\nBuild robust and maintainable object-oriented Python applications and libraries\n\nSteven F. Lott\n\nDusty Phillips\n\nBIRMINGHAM—MUMBAI\n\nPython Object-Oriented Programming Fourth Edition Copyright © 2021 Packt Publishing\n\nAll rights reserved. No part of this book may be reproduced, stored in a retrieval system, or transmitted in any form or by any means, without the prior written permission of the publisher, except in the case of brief quotations embedded in critical articles or reviews.\n\nEvery effort has been made in the preparation of this book to ensure the accuracy of the information presented. However, the information contained in this book is sold without warranty, either express or implied. Neither the authors, nor Packt Publishing or its dealers and distributors, will be held liable for any damages caused or alleged to have been c

In [10]:
text_splitter = CharacterTextSplitter(chunk_size=2000, chunk_overlap=400)

In [11]:
texts = text_splitter.split_documents(documents)

In [12]:
type(texts)

list

In [13]:
len(texts)

798

In [14]:
embeddings = HuggingFaceEmbeddings()

/tmp/ipykernel_9429/3655315981.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/tmp/ipykernel_9429/3655315981.py:1: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/to

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
persist_directory = "vector_db"

In [16]:
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [17]:
retriever = vectordb.as_retriever()

In [19]:
# llm from groq
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

!pip install langchain --upgrade --force-reinstall
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# invoke the qa chain and get a response for user query
query = "what are the function from this pdf "
response = qa_chain.invoke({"query": query})

  Using cached langchain-1.2.12-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain_core-1.2.19-py3-none-any.whl.metadata (4.4 kB)
  Using cached langgraph-1.1.2-py3-none-any.whl.metadata (7.4 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached langsmith-0.7.17-py3-none-any.whl.metadata (15 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.8 kB)
  Using cached langgraph_checkpoint-4.0.1-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_prebuilt-1.0.8-py3-none-any.whl.metadata (5.2 kB)
  U

In [20]:
# invoke the qa chain and get a response for user query
query = "what are the function from this pdf "
response = qa_chain.invoke({"query": query})

In [23]:
# This cell is now empty as its content has been moved to QiOB00AAhowC.

In [21]:
query = "what are the function from this pdf"
response = qa_chain({"query": query})
print(response)

/tmp/ipykernel_9429/3725062613.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = qa_chain({"query": query})


{'query': 'what are the function from this pdf', 'result': 'Based on the provided context, the following functions are mentioned:\n\n1. `fizz(x: int) -> bool`: This function checks if a number `x` is a multiple of 3.\n2. `buzz(x: int) -> bool`: This function checks if a number `x` is a multiple of 5.\n3. `name_or_number(number: int, *tests: Callable[[int], bool]) -> None`: This function takes a number and one or more test functions as arguments. It applies each test function to the number and returns the name of the first test function that returns `True`. If none of the test functions return `True`, it returns the number as a string.\n\nAdditionally, the context mentions a potential function `bazz(x: int) -> bool` that could be defined to check if a number `x` is a multiple of 7, but it is not explicitly defined in the provided code.', 'source_documents': [Document(id='ec865846-d744-442f-9539-221b69cd5aaf', metadata={'source': 'Python Object-Oriented Programming.pdf'}, page_content=">